# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [8]:
class matrix:
    def __init__(self, *args):
        if len(args) == 2:
            n, m = args
            if not isinstance(n, int) or not isinstance(m, int) or n <= 0 or m <= 0:
                raise ValueError("Matrix dimensions must be positive integers")
            self.data = [[0 for j in range(m)] for i in range(n)]

        elif len(args) == 1:
            values = args[0]
            if not isinstance(values, list) or len(values) == 0:
                raise ValueError("Matrix must be a non-empty list of lists")
            if not all(isinstance(row, list) for row in values):
                raise ValueError("Matrix must be a list of lists")
            if len(values[0]) == 0:
                raise ValueError("Matrix rows cannot be empty")

            m = len(values[0])
            if any(len(row) != m for row in values):
                raise ValueError("All rows must have the same number of columns")

            self.data = [row[:] for row in values]

        else:
            raise ValueError("Use matrix(n, m) or matrix(list_of_lists)")

    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key

            if isinstance(i, slice) or isinstance(j, slice):
                if isinstance(i, slice):
                    rows = range(*i.indices(self.shape()[0]))
                else:
                    rows = [i]

                if isinstance(j, slice):
                    columns = range(*j.indices(self.shape()[1]))
                else:
                    columns = [j]

                return matrix([[self.data[r][c] for c in columns] for r in rows])

            return self.data[i][j]

        if isinstance(key, slice):
            return matrix([row[:] for row in self.data[key]])

        return self.data[key]

    def assign(self, other):
        if isinstance(other, matrix):
            values = other.data
        else:
            values = other

        if not isinstance(values, list) or not all(isinstance(row, list) for row in values):
            raise ValueError("Assignment must be from a matrix or list of lists")

        if len(values) != self.shape()[0]:
            raise ValueError("Matrices must have the same size")

        if any(len(row) != self.shape()[1] for row in values):
            raise ValueError("Matrices must have the same size")

        self.data = [row[:] for row in values]

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        n, m = self.shape()
        return matrix([[self.data[i][j] for i in range(n)] for j in range(m)])

    def row(self, n):
        return matrix([self.data[n][:]])

    def column(self, n):
        return matrix([[self.data[i][n]] for i in range(self.shape()[0])])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        return matrix([row[n_0:n_1] for row in self.data[m_0:m_1]])

    def scalarmul(self, c):
        if not isinstance(c, (int, float)):
            raise TypeError("Scalar must be a number")
        return matrix([[c * value for value in row] for row in self.data])

    def add(self, N):
        if not isinstance(N, matrix) or self.shape() != N.shape():
            raise ValueError("Matrices must have the same size")

        n, m = self.shape()
        return matrix([[self.data[i][j] + N.data[i][j]
                        for j in range(m)] for i in range(n)])

    def sub(self, N):
        if not isinstance(N, matrix) or self.shape() != N.shape():
            raise ValueError("Matrices must have the same size")

        n, m = self.shape()
        return matrix([[self.data[i][j] - N.data[i][j]
                        for j in range(m)] for i in range(n)])

    def mat_mult(self, N):
        if not isinstance(N, matrix):
            raise TypeError("Matrix multiplication requires another matrix")

        n, m = self.shape()
        p, q = N.shape()

        if m != p:
            raise ValueError("Matrix dimensions are not compatible for multiplication")

        return matrix([
            [sum(self.data[i][k] * N.data[k][j] for k in range(m))
             for j in range(q)]
            for i in range(n)
        ])

    def element_mult(self, N):
        if not isinstance(N, matrix) or self.shape() != N.shape():
            raise ValueError("Matrices must have the same size")

        n, m = self.shape()
        return matrix([[self.data[i][j] * N.data[i][j]
                        for j in range(m)] for i in range(n)])

    def equals(self, N):
        return isinstance(N, matrix) and self.data == N.data

    def __rmul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        return NotImplemented

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        if isinstance(other, matrix):
            return self.mat_mult(other)
        return NotImplemented

    def __add__(self, other):
        if isinstance(other, matrix):
            return self.add(other)
        return NotImplemented

    def __sub__(self, other):
        if isinstance(other, matrix):
            return self.sub(other)
        return NotImplemented

    def __eq__(self, other):
        return self.equals(other)


def constant(n, m, c):
    return matrix([[float(c) for j in range(m)] for i in range(n)])


def zeros(n, m):
    return constant(n, m, 0)


def ones(n, m):
    return constant(n, m, 1)


def eye(n):
    return matrix([
        [1.0 if i == j else 0.0 for j in range(n)]
        for i in range(n)
    ])


print("Question 1")

M1 = matrix(2, 3)
M2 = matrix([[1, 2, 3],
             [4, 5, 6]])

print(M1.to_list())
print(M2.to_list())
print(M2[0][1])
print(M2[0, 1])

M1.assign(M2)
print(M1.to_list())

M1.assign([[7, 8, 9],
           [10, 11, 12]])
print(M1.to_list())

try:
    matrix([[1, 2], [3, 4, 5]])
except ValueError as e:
    print(e)

try:
    M1.assign([[1, 2], [3, 4]])
except ValueError as e:
    print(e)


print("\nQuestion 2")

M = matrix([[1, 2, 3],
            [4, 5, 6],
            [7, 8, 9]])

print(M.shape())
print(M.transpose().to_list())
print(M.row(1).to_list())
print(M.column(1).to_list())
print(M.to_list())
print(M.block(0, 2, 0, 2).to_list())
print(M[0:2, 1:3].to_list())


print("\nQuestion 3")

print(constant(2, 3, 5).to_list())
print(zeros(2, 3).to_list())
print(ones(2, 3).to_list())
print(eye(3).to_list())


print("\nQuestion 4")

M = matrix([[1, 2],
            [3, 4]])

N = matrix([[5, 6],
            [7, 8]])

print(M.scalarmul(2).to_list())
print(M.add(N).to_list())
print(M.sub(N).to_list())
print(M.mat_mult(N).to_list())
print(M.element_mult(N).to_list())
print(M.equals(N))
print(M.equals(matrix([[1, 2], [3, 4]])))

try:
    M.add(matrix([[1, 2, 3], [4, 5, 6]]))
except ValueError as e:
    print(e)

try:
    M.mat_mult(matrix([[1, 2, 3]]))
except ValueError as e:
    print(e)


print("\nQuestion 5")

print((2 * M).to_list())
print((M * 2).to_list())
print((M + N).to_list())
print((M - N).to_list())
print((M * N).to_list())
print(M == N)

P = matrix([[1, 2],
            [3, 4]])

print(M == P)


print("\nQuestion 6")

A = matrix([[1, 2],
            [3, 4]])

B = matrix([[5, 6],
            [7, 8]])

C = matrix([[2, 0],
            [1, 2]])

I = eye(2)

print("(AB)C =", ((A * B) * C).to_list())
print("A(BC) =", (A * (B * C)).to_list())
print("(AB)C = A(BC):", (A * B) * C == A * (B * C))

print("A(B+C) =", (A * (B + C)).to_list())
print("AB+AC =", (A * B + A * C).to_list())
print("A(B+C) = AB+AC:", A * (B + C) == A * B + A * C)

print("AB =", (A * B).to_list())
print("BA =", (B * A).to_list())
print("AB != BA:", A * B != B * A)

print("AI =", (A * I).to_list())
print("A =", A.to_list())
print("AI = A:", A * I == A)

Question 1
[[0, 0, 0], [0, 0, 0]]
[[1, 2, 3], [4, 5, 6]]
2
2
[[1, 2, 3], [4, 5, 6]]
[[7, 8, 9], [10, 11, 12]]
All rows must have the same number of columns
Matrices must have the same size

Question 2
(3, 3)
[[1, 4, 7], [2, 5, 8], [3, 6, 9]]
[[4, 5, 6]]
[[2], [5], [8]]
[[1, 2, 3], [4, 5, 6], [7, 8, 9]]
[[1, 2], [4, 5]]
[[2, 3], [5, 6]]

Question 3
[[5.0, 5.0, 5.0], [5.0, 5.0, 5.0]]
[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]
[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]

Question 4
[[2, 4], [6, 8]]
[[6, 8], [10, 12]]
[[-4, -4], [-4, -4]]
[[19, 22], [43, 50]]
[[5, 12], [21, 32]]
False
True
Matrices must have the same size
Matrix dimensions are not compatible for multiplication

Question 5
[[2, 4], [6, 8]]
[[2, 4], [6, 8]]
[[6, 8], [10, 12]]
[[-4, -4], [-4, -4]]
[[19, 22], [43, 50]]
False
True

Question 6
(AB)C = [[60, 44], [136, 100]]
A(BC) = [[60, 44], [136, 100]]
(AB)C = A(BC): True
A(B+C) = [[23, 26], [53, 58]]
AB+AC = [[23, 26], [53, 58]]
A(B+C) = AB+